# S11 - Masked Language Models & RAG
## Exercises

### Exercise 1 (Easy)
Use BERT for masked language modeling (fill in the blank).

In [3]:
from transformers import pipeline

# Create a fill-mask pipeline
# Predict: "Paris is the [MASK] of France."

pipe = pipeline("fill-mask", model="bert-base-uncased")
result = pipe("Paris is the [MASK] of France.")

print("\nTop Predictions:\n")
for i, r in enumerate(result, 1):
    print(f"{i}. {r['token_str'].strip()}  (score: {r['score']:.4f})")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 13347.13it/s]
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Top Predictions:

1. capital  (score: 0.9969)
2. heart  (score: 0.0006)
3. center  (score: 0.0004)
4. centre  (score: 0.0003)
5. city  (score: 0.0003)


### Exercise 2 (Easy)
Generate sentence embeddings using a sentence-transformer model.

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The cat sat on the mat.",
    "A dog is playing in the park.",
    "The feline rested on the rug."
]

# Generate embeddings and compute cosine similarities
embeddings = SentenceTransformer('all-MiniLM-L6-v2').encode(sentences)

similarity_matrix = cosine_similarity(embeddings)
print("\nCosine Similarity Matrix:\n")
print(similarity_matrix)

c:\Users\Kike\Programacion\personal\NLP_portfolio_posada_enrique\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kike\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12648


Cosine Similarity Matrix:

[[1.         0.08516441 0.54904425]
 [0.08516441 1.         0.09410812]
 [0.54904425 0.09410812 1.        ]]


### Exercise 3 (Medium)
Build a simple semantic search using FAISS.

In [6]:
import faiss
import numpy as np

documents = [
    "Python is a programming language.",
    "Machine learning uses statistical methods.",
    "Deep learning is a subset of machine learning.",
    "Natural language processing handles text data.",
    "Computer vision processes images."
]

# Create FAISS index and search for "How do neural networks work?"
index = faiss.IndexFlatL2(384)  # Assuming 384-dimensional embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(documents)
index.add(np.array(doc_embeddings))
query = "How do neural networks work?"
query_embedding = model.encode([query])
D, I = index.search(np.array(query_embedding), k=3)
print("\nTop 3 Similar Documents:\n")
for i in I[0]:
    print(f"- {documents[i]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13285.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Top 3 Similar Documents:

- Deep learning is a subset of machine learning.
- Machine learning uses statistical methods.
- Computer vision processes images.


### Exercise 4 (Medium)
Implement a RAG pipeline using TF-IDF retrieval and a local LLM via Ollama.

In the previous exercises you used HuggingFace models for both retrieval and generation. Now you will build a RAG pipeline that connects to a **local LLM running on Ollama** (as you learned in S10) for the generation step. For retrieval, you will use a simple TF-IDF approach instead of FAISS — this removes the dependency on sentence-transformers and keeps the focus on the local LLM integration.

The knowledge base consists of short paragraphs about NLP concepts from this course.

**Task:**
1. Create a knowledge base with at least 6 paragraphs about NLP topics (tokenization, embeddings, attention, transformers, sentiment analysis, named entity recognition)
2. Implement a `retrieve(query, documents, top_k=2)` function that uses `TfidfVectorizer` and `cosine_similarity` from sklearn to find the most relevant documents
3. Implement a `generate_answer(query, context)` function that sends the retrieved context and the question to Ollama's `/api/generate` endpoint and returns the model's answer
4. Test your RAG pipeline with at least 3 different questions about NLP topics
5. For each question, print: the retrieved documents (with similarity scores), the model's answer, and whether the answer correctly uses the retrieved context

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"

# --- Knowledge Base ---
documents = [
    "Tokenization is the process of splitting raw text into smaller meaningful units called tokens. These tokens can be words, subwords, or characters depending on the model or task. Tokenization is a fundamental preprocessing step in NLP because it converts unstructured text into a format that machine learning models can process efficiently. Modern transformer models often use subword tokenization methods like Byte Pair Encoding (BPE) or WordPiece to handle rare and unknown words effectively.",

    "Stemming is a text normalization technique that reduces words to their root form by removing suffixes and prefixes using heuristic rules. For example, words like 'running', 'runner', and 'ran' may be reduced to 'run'. However, stemming does not always produce linguistically correct words, as it focuses on rule-based chopping rather than understanding meaning. Despite its simplicity, stemming is still useful in information retrieval systems where speed matters more than linguistic accuracy.",

    "Lemmatization is a more advanced form of text normalization that reduces words to their dictionary base form, known as the lemma. Unlike stemming, lemmatization considers the context and part of speech of a word to ensure the output is linguistically correct. For example, 'better' is converted to 'good' and 'running' becomes 'run'. It is commonly used in applications where semantic accuracy is important, such as question answering and information extraction.",

    "Attention mechanism is a core component of transformer models that allows the model to focus on the most relevant parts of an input sequence when making predictions. Instead of processing text sequentially like traditional RNNs, attention assigns different weights to all words in a sentence, determining which tokens are most important for understanding context. This enables transformers to capture long-range dependencies and relationships between words more effectively, making them highly powerful for tasks such as translation, summarization, and question answering.",

    "Word embeddings are dense vector representations of words that capture semantic meaning based on their usage in large text corpora. Unlike traditional one-hot encoding, embeddings place similar words closer together in a high-dimensional vector space. Models like Word2Vec, GloVe, and FastText helped establish this concept, while modern transformer-based models generate contextual embeddings where a word's meaning depends on surrounding words. These representations are essential for tasks like semantic search and machine translation.",

    "Sentiment analysis is the process of determining the emotional tone behind a piece of text, typically classifying it as positive, negative, or neutral. It is widely used in social media monitoring, customer feedback analysis, and market research. More advanced sentiment models can detect emotions such as anger, joy, or sadness. Transformer-based models significantly improve sentiment analysis by understanding context, sarcasm, and nuanced language patterns better than traditional machine learning approaches."
]

# --- Retrieval Function ---
def retrieve(query, documents, top_k=2):
    # Use TF-IDF + cosine similarity to find the top_k most relevant documents
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, tfidf_matrix)
    top_indices = similarities.argsort()[0][-top_k:][::-1]
    return [documents[i] for i in top_indices]

# --- Generation Function ---
def generate_answer(query, context):
    # Send the query and retrieved context to Ollama's /api/generate endpoint
    # Return the model's answer
    prompt = f"""You are an expert in NLP. Concisely answer the following question based on the provided context.

Context:
{context}

Question: {query}
"""

    ollama_response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False
    }
)
    ollama_text = ollama_response.json()["response"]
    return ollama_text

# --- Test the RAG Pipeline ---
questions = [
    "What is the attention mechanism in transformers?",
    "How does tokenization work in NLP?",
    "What is sentiment analysis used for?",
]

# For each question: retrieve context, generate answer, print results
for q in questions:
    retrieved_docs = retrieve(q, documents)
    context = "\n\n".join(retrieved_docs)
    answer = generate_answer(q, context)
    print(f"\nQuestion: {q}\n")
    print(f"Retrieved Context:\n{context}\n")
    print(f"Generated Answer:\n{answer}\n")


Question: What is the attention mechanism in transformers?

Retrieved Context:
Attention mechanism is a core component of transformer models that allows the model to focus on the most relevant parts of an input sequence when making predictions. Instead of processing text sequentially like traditional RNNs, attention assigns different weights to all words in a sentence, determining which tokens are most important for understanding context. This enables transformers to capture long-range dependencies and relationships between words more effectively, making them highly powerful for tasks such as translation, summarization, and question answering.

Lemmatization is a more advanced form of text normalization that reduces words to their dictionary base form, known as the lemma. Unlike stemming, lemmatization considers the context and part of speech of a word to ensure the output is linguistically correct. For example, 'better' is converted to 'good' and 'running' becomes 'run'. It is common

### Exercise 5 (Hard)
Build a RAG-powered Q&A system with answer quality evaluation using a local LLM.

In this exercise, you will extend the RAG pipeline from Exercise 4 into a more complete Q&A system. You will create a set of questions with known (ground truth) answers, run them through your RAG pipeline, and evaluate the quality of the generated answers — both automatically and manually.

This simulates a real-world scenario where you need to assess whether a RAG system is reliable enough for production use.

**Task:**
1. Expand the knowledge base from Exercise 4 to at least 10 documents covering a broader range of NLP topics (add documents about word embeddings, RNNs, LSTMs, BERT, text classification, language modeling, etc.)
2. Create an evaluation dataset: a list of at least 8 question-answer pairs where the answer can be found in the knowledge base. Include at least 2 questions whose answer is **not** in the knowledge base (to test how the system handles missing information)
3. Run each question through the RAG pipeline and collect the generated answers
4. Implement two evaluation methods:
   - **Automatic:** Use Ollama itself as a judge — send the question, ground truth answer, and generated answer to the model and ask it to rate the answer on a scale of 1-5 (faithfulness to context, correctness, completeness)
   - **Manual:** For each answer, print the question, retrieved context, generated answer, and ground truth side by side so you can inspect the results
5. Print a summary with the average score across all questions and identify which questions the system struggled with

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"

# --- Expanded Knowledge Base ---
documents = [
    "Tokenization is the process of splitting raw text into smaller meaningful units called tokens such as words or subwords. It is a fundamental preprocessing step in NLP that converts text into a format models can process. Modern systems often use subword tokenization like Byte Pair Encoding (BPE) or WordPiece to handle rare words effectively.",

    "Word embeddings are dense vector representations of words that capture semantic meaning based on context. Unlike one-hot encoding, embeddings place similar words closer in vector space. Popular methods include Word2Vec, GloVe, and FastText, while transformer models produce contextual embeddings that change meaning based on surrounding words.",

    "Recurrent Neural Networks (RNNs) are neural architectures designed to process sequential data by maintaining a hidden state that captures information from previous time steps. They are useful for tasks like language modeling and time series prediction but suffer from vanishing gradient problems when handling long sequences.",

    "Long Short-Term Memory networks (LSTMs) are a type of RNN designed to overcome the vanishing gradient problem. They use memory cells and gating mechanisms (input, forget, and output gates) to control the flow of information, allowing them to capture long-range dependencies in sequential data.",

    "Transformers are deep learning models that rely on self-attention mechanisms instead of recurrence. They process entire sequences in parallel and are highly effective for NLP tasks such as translation, summarization, and question answering. They form the basis of models like BERT and GPT.",

    "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based model that reads text in both directions (left-to-right and right-to-left). It is pre-trained using masked language modeling and next sentence prediction, making it highly effective for understanding context in NLP tasks.",

    "Text classification is the task of assigning predefined categories to text, such as spam detection or sentiment analysis. It can be performed using traditional machine learning models or modern deep learning architectures like transformers, depending on dataset size and complexity.",

    "Language modeling is the task of predicting the next word or sequence of words in a sentence. It is a foundational task in NLP used in applications like autocomplete, speech recognition, and machine translation. Models can be statistical (n-grams) or neural (RNNs, Transformers).",

    "Attention mechanism allows models to focus on the most relevant parts of an input sequence when generating outputs. It assigns weights to different tokens, enabling better handling of long-range dependencies and improving performance in translation and summarization tasks.",

    "Named Entity Recognition (NER) is the task of identifying and classifying entities in text such as people, organizations, locations, and dates. It is widely used in information extraction, search engines, and building knowledge graphs."
]

# --- Evaluation Dataset ---
eval_data = [
    ("What is tokenization?", "Tokenization is the process of splitting text into smaller units called tokens."),
    ("What are word embeddings?", "Word embeddings are dense vector representations of words that capture semantic meaning."),
    ("What is the difference between RNNs and LSTMs?", "LSTMs improve RNNs by solving the vanishing gradient problem using gating mechanisms."),
    ("What is a transformer model?", "Transformers are deep learning models that use self-attention instead of recurrence."),
    ("What is BERT used for?", "BERT is used for understanding context in NLP using bidirectional transformers."),
    ("What is language modeling?", "Language modeling is the task of predicting the next word in a sequence."),
    ("What is attention mechanism?", "Attention mechanism helps models focus on relevant parts of input sequences."),
    ("What is text classification?", "Text classification assigns predefined categories to text."),
    
    # Out-of-domain questions (NOT in knowledge base)
    ("What is reinforcement learning?", "Not in knowledge base"),
    ("What is computer vision?", "Not in knowledge base")
]

# --- Reuse or redefine retrieve() and generate_answer() from Exercise 4 ---
# --- Retrieval Function ---
def retrieve(query, documents, top_k=2):
    # Use TF-IDF + cosine similarity to find the top_k most relevant documents
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, tfidf_matrix)
    top_indices = similarities.argsort()[0][-top_k:][::-1]
    return [documents[i] for i in top_indices]

# --- Generation Function ---
def generate_answer(query, context):
    # Send the query and retrieved context to Ollama's /api/generate endpoint
    # Return the model's answer
    prompt = f"""You are an expert in NLP. Concisely answer the following question based on the provided context.

Context:
{context}

Question: {query}
"""

    ollama_response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False
    }
)
    ollama_text = ollama_response.json()["response"]
    return ollama_text


# --- LLM-as-Judge Evaluation ---
def evaluate_answer(question, ground_truth, generated_answer):
    # Send to Ollama: ask it to rate the generated answer vs ground truth (1-5)
    # Return the score
    prompt = f"""You are an expert in NLP. Rate the following answer to the question on a scale of 1 to 5, where 5 is a perfect answer and 1 is a very poor answer.

Question: {question}
Ground Truth: {ground_truth}
Generated Answer: {generated_answer}
Please provide only the score (1-5) without any explanation.
"""

    ollama_response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False
    }
)
    ollama_text = ollama_response.json()["response"].strip()
    return ollama_text

worst_performing = []
total_score = 0

# --- Run Evaluation ---
# For each question: retrieve, generate, evaluate, print results
for q, gt in eval_data:
    retrieved_docs = retrieve(q, documents)
    context = "\n\n".join(retrieved_docs)
    generated_answer = generate_answer(q, context)
    score = evaluate_answer(q, gt, generated_answer)
    
    total_score += int(score)
    worst_performing.append((q, gt, generated_answer, score))

    print(f"\nQuestion: {q}\n")
    print(f"Ground Truth: {gt}\n")
    print(f"Generated Answer: {generated_answer}\n")
    print(f"Score (1-5): {score}\n")

# --- Print Summary ---
# Average score, worst-performing questions, observations
average_score = total_score / len(eval_data)
worst_performing.sort(key=lambda x: int(x[3]))  # Sort by score

print(f"\nAverage Score: {average_score:.2f}\n")
print("Worst Performing Questions:\n")
for q, gt, ans, score in worst_performing[:3]:  # Print top 3 worst
    print(f"Question: {q}\nGround Truth: {gt}\nGenerated Answer: {ans}\nScore: {score}\n")


Question: What is tokenization?

Ground Truth: Tokenization is the process of splitting text into smaller units called tokens.

Generated Answer: Tokenization is the process of splitting text into smaller units like words or subwords.


Score (1-5): 4


Question: What are word embeddings?

Ground Truth: Word embeddings are dense vector representations of words that capture semantic meaning.

Generated Answer: Word embeddings are dense vector representations of words capturing semantic meaning based on context, unlike one-hot encoding.

Score (1-5): 1


Question: What is the difference between RNNs and LSTMs?

Ground Truth: LSTMs improve RNNs by solving the vanishing gradient problem using gating mechanisms.

Generated Answer: RNNs are a basic sequential model, while LSTMs are a specialized type of RNN enhanced with memory cells and gating mechanisms to better handle long-range dependencies.

Score (1-5): 1


Question: What is a transformer model?

Ground Truth: Transformers are deep l

Although there were two questions whose answers were not present in the knowledge base, the model still produced reasonable responses given the lack of retrieved context. However, both of these were scored as 1, which suggests that the evaluation mechanism may be overly strict. In several cases, responses that were semantically very close to the ground truth also received low scores (just 1 point), indicating that the judge is sensitive to minor differences in wording rather than overall meaning. The average score of 2.5 indicates moderate performance, suggesting that the RAG + evaluation system is functional but could benefit from a more robust and semantically aware scoring method.